In [2]:
# imports
import pandas as pd
import numpy as np

In [4]:
file_path = r"C:/Users/Peter/OneDrive/Desktop/SCADA/WFAnalysis/dataset/CARE_To_Compare/Wind Farm A/datasets/68.csv"

df = pd.read_csv(file_path, sep=";")

print("Dataset loaded successfully.")
# df.head()
# df.tail()
pd.set_option('display.max_rows', None)
pd.set_option('display.max_columns', None)

list_of_ids = [49253]
anomaly = df.loc[list_of_ids]
print(anomaly)



Dataset loaded successfully.
                time_stamp  asset_id     id train_test  status_type_id  \
49253  2023-07-08 13:20:00        11  49253      train               0   

       sensor_0_avg  sensor_1_avg  sensor_2_avg  wind_speed_3_avg  \
49253          24.0         265.7           3.0               5.9   

       wind_speed_4_avg  wind_speed_3_max  wind_speed_3_min  wind_speed_3_std  \
49253               5.9              22.1               1.3               1.6   

       sensor_5_avg  sensor_5_max  sensor_5_min  sensor_5_std  sensor_6_avg  \
49253          -1.5           0.5          -2.5           0.7          35.0   

       sensor_7_avg  sensor_8_avg  sensor_9_avg  sensor_10_avg  sensor_11_avg  \
49253          42.0          93.0          45.0           39.0           54.0   

       sensor_12_avg  sensor_13_avg  sensor_14_avg  sensor_15_avg  \
49253           48.0           39.0           45.0           65.0   

       sensor_16_avg  sensor_17_avg  sensor_18_avg  sensor_

In [12]:
import pandas as pd
import numpy as np

def query_events_by_id_range(
    scada_csv_path: str,
    event_info_path: str,
    row_id_col: str = "id",              
    event_id_col: str = "event_id",
    start_id_col: str = "event_start_id",
    end_id_col: str = "event_end_id",
    sep_scada: str = ";",
    sep_events: str = ";",
):
    # Load
    df = pd.read_csv(scada_csv_path, sep=sep_scada)
    events = pd.read_csv(event_info_path, sep=sep_events)

    # Keep only needed columns + clean types
    df[row_id_col] = pd.to_numeric(df[row_id_col], errors="coerce")
    events[start_id_col] = pd.to_numeric(events[start_id_col], errors="coerce")
    events[end_id_col]   = pd.to_numeric(events[end_id_col], errors="coerce")

    df = df.dropna(subset=[row_id_col]).copy()
    events = events.dropna(subset=[event_id_col, start_id_col, end_id_col]).copy()

    # Ensure integer IDs (optional)
    df[row_id_col] = df[row_id_col].astype(np.int64)
    events[start_id_col] = events[start_id_col].astype(np.int64)
    events[end_id_col]   = events[end_id_col].astype(np.int64)

    # Sort once (needed for fast searchsorted)
    df = df.sort_values(row_id_col).reset_index(drop=True)
    ids = df[row_id_col].to_numpy()

    pieces = []
    for ev in events.itertuples(index=False):
        ev_id = getattr(ev, event_id_col)
        start = getattr(ev, start_id_col)
        end   = getattr(ev, end_id_col)

        # Handle reversed ranges just in case
        if end < start:
            start, end = end, start

        left = np.searchsorted(ids, start, side="left")
        right = np.searchsorted(ids, end, side="right")  # inclusive end
        if right > left:
            chunk = df.iloc[left:right].copy()
            chunk[event_id_col] = ev_id
            pieces.append(chunk)

    return pd.concat(pieces, ignore_index=True) if pieces else df.iloc[0:0].copy()


# Example:
anomaly_df = query_events_by_id_range(
    scada_csv_path=r"C:\Users\Peter\OneDrive\Desktop\SCADA\WFAnalysis\dataset\CARE_To_Compare\Wind Farm A\datasets\68.csv",
    event_info_path=r"C:\Users\Peter\OneDrive\Desktop\SCADA\WFAnalysis\dataset\CARE_To_Compare\Wind Farm A\event_info.csv",
    row_id_col="id",   
    sep_scada=";",
    sep_events=";"
)

print(anomaly_df.head(21))
print("rows:", len(anomaly_df), "events:", anomaly_df["event_id"].nunique() if "event_id" in anomaly_df.columns else 0)

             time_stamp  asset_id     id  train_test  status_type_id  \
0   2023-07-28 13:20:00        11  52063  prediction               4   
1   2023-07-28 13:30:00        11  52064  prediction               4   
2   2023-07-28 13:40:00        11  52065  prediction               4   
3   2023-07-28 13:50:00        11  52066  prediction               4   
4   2023-07-28 14:00:00        11  52067  prediction               4   
5   2023-07-28 14:10:00        11  52068  prediction               4   
6   2023-07-28 14:20:00        11  52069  prediction               4   
7   2023-07-28 14:30:00        11  52070  prediction               4   
8   2023-07-28 14:40:00        11  52071  prediction               4   
9   2023-07-28 14:50:00        11  52072  prediction               4   
10  2023-07-28 15:00:00        11  52073  prediction               4   
11  2023-07-28 15:10:00        11  52074  prediction               4   
12  2023-07-28 15:20:00        11  52075  prediction            